# ECG Classification — Phase 2: Stage 1 (Improved Normal vs Non-Normal)

**What changed vs the original:**

| Area | Original | Improved |
|------|----------|----------|
| Model | TCN 3-block, 256 ch | TCN-ResNet hybrid, 5 dilated blocks, 512 ch, multi-scale pooling |
| Features | Raw signal only | Raw signal + NeuroKit2 features (late fusion) |
| Loss | CrossEntropyLoss + fixed class weights | Asymmetric Focal Loss — penalises missed Normals harder |
| Augmentation | None at training time | Online: random crop-shift, amplitude scale, time-mask |
| Sampling | Standard DataLoader | `WeightedRandomSampler` — guarantees balanced batches |
| Optimiser | RAdam, flat LR | AdamW + cosine-with-warmup schedule |
| Regularisation | Dropout 0.2 | Dropout 0.3 + stochastic depth (LayerDrop) |
| Threshold | Swept 0.30–0.70 | Fine-grained sweep 0.25–0.75, saved for inference |

**Expected gains:** macro F1 0.88 → ~0.93–0.95, with Normal F1 ≥ 0.90


## 1 · Imports

In [ ]:
import os, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())


PyTorch : 2.10.0+cu128
CUDA    : True


## 2 · Device and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_PATH = "/content/drive/MyDrive"
DATA_PATH = os.path.join(BASE_PATH, "ECG_Hierarchical")
CKPT_DIR  = os.path.join(BASE_PATH, "checkpoints_stage1_v2")
os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_MODEL   = os.path.join(CKPT_DIR, "best_model.pt")
CKPT_STATE   = os.path.join(CKPT_DIR, "train_state.json")
HISTORY_PATH = os.path.join(CKPT_DIR, "history.json")

print("Device        :", DEVICE)
print("Data path     :", DATA_PATH)
print("Checkpoint dir:", CKPT_DIR)


Device        : cuda
Data path     : /content/drive/MyDrive/ECG_Hierarchical
Checkpoint dir: /content/drive/MyDrive/checkpoints_stage1_v2


## 3 · Hyperparameters

In [ ]:
WINDOW_SIZE  = 4500
BATCH_SIZE   = 64
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 50
PATIENCE     = 20      # more patience — cosine LR needs room to recover
VAL_FRAC     = 0.15
SEED         = 42
WARMUP_PCT   = 0.05    # fraction of total steps used for LR warm-up

# Asymmetric Focal Loss parameters
# gamma_neg > gamma_pos  →  harder penalty on false negatives for Normal class
AFL_GAMMA_POS = 1.0
AFL_GAMMA_NEG = 3.0

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


## 4 · Load data

In [ ]:
X_sig  = np.load(os.path.join(DATA_PATH, "X_train_sig.npy"),  mmap_mode="r")
X_feat = np.load(os.path.join(DATA_PATH, "X_train_feat.npy"), mmap_mode="r")
y_bin  = np.load(os.path.join(DATA_PATH, "y_train_binary.npy"))

print("X_sig  :", X_sig.shape,  " dtype:", X_sig.dtype)
print("X_feat :", X_feat.shape, " dtype:", X_feat.dtype)
print("y_bin  :", y_bin.shape)
n_normal    = (y_bin == 0).sum()
n_nonnormal = (y_bin == 1).sum()
print(f"Class 0 (Normal)   : {n_normal}  ({n_normal/len(y_bin)*100:.1f}%)")
print(f"Class 1 (NonNormal): {n_nonnormal}  ({n_nonnormal/len(y_bin)*100:.1f}%)")
print(f"Imbalance ratio    : {n_nonnormal/n_normal:.2f}×")


X_sig  : (31271, 4500)  dtype: float32
X_feat : (31271, 39)  dtype: float32
y_bin  : (31271,)
Class 0 (Normal)   : 12895  (41.2%)
Class 1 (NonNormal): 18376  (58.8%)
Imbalance ratio    : 1.43×


## 5 · Dataset with online augmentation

Three lightweight transforms applied **only during training**, per-sample at load time:

- **Amplitude scale** — multiply signal by U(0.85, 1.15); preserves morphology, varies amplitude
- **Time shift** — circular-roll by up to ±2 % of window; prevents position bias
- **Time mask** — zero out one random 5 % segment; forces the model to use context, not a single spike


In [ ]:
class ECGDataset(Dataset):
    """
    Returns (signal_tensor, feat_tensor, label).
    If augment=True, applies random online transforms.
    """
    def __init__(self, X_sig, X_feat, y, indices, augment=False):
        # Copy selected rows into RAM to avoid repeated mmap seeks
        self.X_sig  = np.array(X_sig[indices],  dtype=np.float32)
        self.X_feat = np.array(X_feat[indices],  dtype=np.float32)
        self.y      = y[indices].astype(np.int64)
        self.augment = augment
        self.n      = len(indices)
        self.win    = self.X_sig.shape[1]

    def __len__(self):
        return self.n

    def _augment(self, sig):
        sig = sig.copy()
        # 1. Amplitude scale
        if random.random() < 0.5:
            sig *= random.uniform(0.85, 1.15)
        # 2. Time shift (circular roll up to ±2%)
        if random.random() < 0.5:
            shift = random.randint(-int(0.02 * self.win), int(0.02 * self.win))
            sig = np.roll(sig, shift)
        # 3. Time mask: zero one 5% segment
        if random.random() < 0.5:
            mlen  = int(0.05 * self.win)
            start = random.randint(0, self.win - mlen)
            sig[start : start + mlen] = 0.0
        return sig

    def __getitem__(self, idx):
        sig  = self.X_sig[idx]
        feat = self.X_feat[idx]
        if self.augment:
            sig = self._augment(sig)
        return (torch.tensor(sig,  dtype=torch.float32),
                torch.tensor(feat, dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.long))


## 6 · Build DataLoaders with balanced sampling

In [ ]:
def make_loaders(X_sig, X_feat, y_bin, val_frac, batch_size, seed):
    rng   = np.random.default_rng(seed)
    idx   = np.arange(len(y_bin))
    rng.shuffle(idx)
    n_val = int(val_frac * len(idx))
    val_idx, train_idx = idx[:n_val], idx[n_val:]

    train_ds = ECGDataset(X_sig, X_feat, y_bin, train_idx, augment=True)
    val_ds   = ECGDataset(X_sig, X_feat, y_bin, val_idx,   augment=False)

    # WeightedRandomSampler: every batch is ~50/50 Normal / NonNormal
    train_labels = y_bin[train_idx]
    class_counts = np.bincount(train_labels)
    weights      = 1.0 / class_counts[train_labels]
    sampler      = WeightedRandomSampler(
        weights=torch.from_numpy(weights).float(),
        num_samples=len(train_labels),
        replacement=True
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=sampler, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0, pin_memory=False)
    return train_loader, val_loader, len(train_idx), len(val_idx)

train_loader, val_loader, n_train, n_val = make_loaders(
    X_sig, X_feat, y_bin, VAL_FRAC, BATCH_SIZE, SEED)

print("Train windows :", n_train)
print("Val windows   :", n_val)
print("Train batches :", len(train_loader), " (balanced via WeightedRandomSampler)")
print("Val batches   :", len(val_loader))


Train windows : 26581
Val windows   : 4690
Train batches : 416  (balanced via WeightedRandomSampler)
Val batches   : 74


## 7 · Model — TCN-ResNet Hybrid with Feature Fusion

**Architecture overview:**

```
Raw signal (4500,)
    │
    ├── [Input conv]  1 → 64 ch, kernel 15
    │
    ├── [TCN block 1] dilation=1  64  → 64
    ├── [TCN block 2] dilation=2  64  → 128
    ├── [TCN block 3] dilation=4  128 → 256
    ├── [TCN block 4] dilation=8  256 → 512
    ├── [TCN block 5] dilation=16 512 → 512
    │
    ├── [SE-1D attention] 512 ch
    │
    ├── [Multi-scale pooling]
    │       ├── AvgPool → 512
    │       └── MaxPool → 512
    │   → concat → 1024
    │
    ├── [Signal head] 1024 → 256
    │
NeuroKit2 features (n_feat,)
    └── [Feature MLP] n_feat → 64 → 64
    │
    └── [Fusion head] 256 + 64 → 128 → 2
```

**Key improvements:**
- 5 dilated TCN blocks (receptive field covers the full 4500-sample window)
- Multi-scale pooling (avg + max) captures both mean rhythm and peak events
- Late fusion of NeuroKit2 features — adds global HRV context the signal alone can't capture
- Stochastic depth (LayerDrop) — randomly skips entire TCN blocks during training → strong regulariser


In [ ]:
# ── SE-1D attention block ─────────────────────────────────────────────────
class SE1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(channels, max(channels // reduction, 4)), nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.gate(x).unsqueeze(-1)


# ── Dilated TCN block with weight-norm + residual ─────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.3):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.utils.weight_norm(
                nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=pad)),
            nn.PReLU(),
            nn.Dropout(dropout),
            nn.utils.weight_norm(
                nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad)),
            nn.PReLU(),
            nn.Dropout(dropout),
        )
        self.proj = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.trim_len = None   # set dynamically in forward

    def forward(self, x):
        out = self.net(x)
        out = out[:, :, :x.size(2)]   # causal trim
        return out + self.proj(x)


# ── Feature MLP for NeuroKit2 features ────────────────────────────────────
class FeatureMLP(nn.Module):
    def __init__(self, n_feat, hidden=64, out=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out),   nn.LayerNorm(out),    nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)


# ── Full Stage-1 model ─────────────────────────────────────────────────────
class Stage1Model(nn.Module):
    def __init__(self, n_feat, dropout=0.3, layer_drop_p=0.1):
        super().__init__()
        self.layer_drop_p = layer_drop_p

        # Input projection
        self.input_conv = nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(1, 64, kernel_size=15, padding=7)),
            nn.PReLU(),
        )

        # 5 TCN blocks with exponentially growing dilation
        self.blocks = nn.ModuleList([
            TCNBlock(64,  64,  dilation=1,  dropout=dropout),
            TCNBlock(64,  128, dilation=2,  dropout=dropout),
            TCNBlock(128, 256, dilation=4,  dropout=dropout),
            TCNBlock(256, 512, dilation=8,  dropout=dropout),
            TCNBlock(512, 512, dilation=16, dropout=dropout),
        ])

        self.se = SE1D(512)

        # Multi-scale pooling: avg + max → concat → 1024
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)

        # Signal head: 1024 → 256
        self.sig_head = nn.Sequential(
            nn.Linear(1024, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
        )

        # Feature branch
        self.feat_mlp = FeatureMLP(n_feat, hidden=64, out=64, dropout=dropout)

        # Fusion classifier: 256 + 64 → 128 → 2
        self.fusion = nn.Sequential(
            nn.Linear(256 + 64, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, sig, feat):
        # sig : (B, 4500)  feat : (B, n_feat)
        x = sig.unsqueeze(1)           # (B, 1, 4500)
        x = self.input_conv(x)         # (B, 64, 4500)

        for block in self.blocks:
            # Stochastic depth: randomly skip block during training
            if self.training and torch.rand(1).item() < self.layer_drop_p:
                # If channel dims differ, still need to project
                if hasattr(block, 'proj') and not isinstance(block.proj, nn.Identity):
                    x = block.proj(x)
            else:
                x = block(x)

        x = self.se(x)                            # (B, 512, L)
        avg = self.avg_pool(x).squeeze(-1)         # (B, 512)
        mx  = self.max_pool(x).squeeze(-1)         # (B, 512)
        x   = torch.cat([avg, mx], dim=1)          # (B, 1024)
        x   = self.sig_head(x)                     # (B, 256)

        f   = self.feat_mlp(feat)                  # (B, 64)
        out = self.fusion(torch.cat([x, f], dim=1))  # (B, 2)
        return out


## 8 · Instantiate model, loss, optimiser, scheduler

In [ ]:
n_feat = X_feat.shape[1]
model  = Stage1Model(n_feat=n_feat, dropout=0.3, layer_drop_p=0.10).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Total parameters    : 9,961,901
Trainable parameters: 9,961,901


### Asymmetric Focal Loss

Standard Focal Loss uses the same `gamma` for both classes. **Asymmetric Focal Loss** uses:
- `gamma_pos` (low, e.g. 1.0) for the majority class — moderate down-weighting of easy non-normal samples
- `gamma_neg` (high, e.g. 3.0) for the minority class — aggressively down-weights easy Normal samples, forcing the model to focus on hard-to-classify Normals

This directly attacks the problem of everything being predicted as NonNormal.


In [ ]:
class AsymmetricFocalLoss(nn.Module):
    """
    Multi-class Asymmetric Focal Loss.
    gamma_neg > gamma_pos penalises missed Normal (minority) examples harder.
    """
    def __init__(self, gamma_pos=1.0, gamma_neg=3.0, eps=1e-6):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.eps       = eps

    def forward(self, logits, targets):
        # logits : (B, 2)   targets : (B,) int
        log_probs = F.log_softmax(logits, dim=1)          # (B, 2)
        probs     = log_probs.exp()

        # one-hot
        B = targets.size(0)
        onehot = torch.zeros_like(probs).scatter_(1, targets.unsqueeze(1), 1)  # (B,2)

        # p_t per sample per class
        p_t = (probs * onehot).sum(1)                     # (B,)

        # asymmetric gamma
        gamma = torch.where(targets == 0,
                            torch.tensor(self.gamma_neg, device=logits.device),
                            torch.tensor(self.gamma_pos, device=logits.device))

        focal_weight = (1.0 - p_t + self.eps) ** gamma
        loss = -(focal_weight * (probs * onehot + self.eps).log().sum(1))
        return loss.mean()


criterion = AsymmetricFocalLoss(gamma_pos=AFL_GAMMA_POS, gamma_neg=AFL_GAMMA_NEG)
print("Loss: AsymmetricFocalLoss  gamma_pos={:.1f}  gamma_neg={:.1f}".format(
    AFL_GAMMA_POS, AFL_GAMMA_NEG))


Loss: AsymmetricFocalLoss  gamma_pos=1.0  gamma_neg=3.0


In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps      = MAX_EPOCHS * steps_per_epoch

scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=total_steps,
    pct_start=WARMUP_PCT,
    anneal_strategy="cos",
    div_factor=10.0,       # start LR = max_lr / 10
    final_div_factor=100.0 # end   LR = max_lr / 1000
)

amp_scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))

print(f"Optimiser : AdamW  lr={LR}  weight_decay={WEIGHT_DECAY}")
print(f"Scheduler : OneCycleLR  total_steps={total_steps}  warmup={WARMUP_PCT*100:.0f}%")


Optimiser : AdamW  lr=0.0003  weight_decay=0.0001
Scheduler : OneCycleLR  total_steps=20800  warmup=5%


## 9 · Checkpoint helpers

In [ ]:
def save_checkpoint(model, optimizer, scheduler, amp_scaler,
                    epoch, best_val_f1, patience_ctr, history):
    torch.save(model.state_dict(), CKPT_MODEL)
    state = {
        "epoch":          epoch,
        "best_val_f1":    best_val_f1,
        "patience_ctr":   patience_ctr,
        "scheduler_last_step": scheduler.last_epoch,
    }
    with open(CKPT_STATE,   "w") as f: json.dump(state, f)
    with open(HISTORY_PATH, "w") as f: json.dump(history, f)

def load_checkpoint(model, optimizer, scheduler):
    if not os.path.exists(CKPT_MODEL) or not os.path.exists(CKPT_STATE):
        print("No checkpoint — starting fresh.")
        return 0, 0.0, 0, {"train": [], "val": []}
    model.load_state_dict(torch.load(CKPT_MODEL, map_location=DEVICE))
    with open(CKPT_STATE) as f:
        state = json.load(f)
    for pg in optimizer.param_groups:
        pg["lr"] = state.get("lr", LR)

    # Advance the OneCycleLR to the correct position by replaying last_epoch
    # dummy steps.  We call scheduler.step() directly (not via optimizer) so
    # no "step before optimizer" warning is raised here.
    last_step = state.get("scheduler_last_step", 0)
    if last_step > 0:
        # Directly set last_epoch on the scheduler to avoid re-running the
        # optimizer, which would trigger the ordering warning.
        scheduler.last_epoch = last_step - 1   # -1 because step() increments first
        scheduler.step()

    history = {"train": [], "val": []}
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)
    print(f"Resumed from epoch {state['epoch']+1}  best val F1: {state['best_val_f1']:.4f}  "
          f"scheduler step: {last_step}")
    return state["epoch"] + 1, state["best_val_f1"], state["patience_ctr"], history


## 10 · Metrics helpers

In [ ]:
def compute_metrics(trues, preds, losses):
    trues, preds = np.array(trues), np.array(preds)
    loss  = float(np.mean(losses))
    acc   = float(np.mean(trues == preds))
    prec  = precision_score(trues, preds, average="macro", zero_division=0)
    rec   = recall_score(trues,    preds, average="macro", zero_division=0)
    f1    = f1_score(trues,        preds, average="macro", zero_division=0)
    f1_N  = f1_score(trues, preds, labels=[0], average="macro", zero_division=0)
    f1_NN = f1_score(trues, preds, labels=[1], average="macro", zero_division=0)
    return {"loss": loss, "acc": acc, "prec": prec, "rec": rec,
            "f1": f1, "f1_N": f1_N, "f1_NN": f1_NN}

def print_epoch(epoch, total, tr, vl, lr, overfit):
    flag = "  *** OVERFIT ***" if overfit else ""
    print(
        f"Ep {epoch:>3}/{total}  lr={lr:.2e}"
        f"  | TR  loss={tr['loss']:.4f}  F1={tr['f1']:.4f}"
        f"  F1_N={tr['f1_N']:.4f}  F1_NN={tr['f1_NN']:.4f}"
        f"  | VL  loss={vl['loss']:.4f}  F1={vl['f1']:.4f}"
        f"  F1_N={vl['f1_N']:.4f}  F1_NN={vl['f1_NN']:.4f}"
        f"{flag}"
    )


## 11 · Train and validation epoch functions

In [ ]:
def run_train_epoch(model, loader, optimizer, criterion, scheduler, amp_scaler):
    model.train()
    all_preds, all_trues, all_losses = [], [], []

    for batch_idx, (x_sig, x_feat, y) in enumerate(tqdm(loader, desc="Train", leave=False)):
        x_sig  = x_sig.to(DEVICE)
        x_feat = x_feat.to(DEVICE)
        y      = y.to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE == "cuda")):
            logits = model(x_sig, x_feat)
            loss   = criterion(logits, y)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        prev_scale = amp_scaler.get_scale()
        amp_scaler.step(optimizer)
        amp_scaler.update()
        if amp_scaler.get_scale() == prev_scale:
            scheduler.step()

        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_trues.extend(y.cpu().numpy())
        all_losses.append(loss.item())

    return compute_metrics(all_trues, all_preds, all_losses)


def run_val_epoch(model, loader, criterion):
    model.eval()
    all_preds, all_trues, all_losses, all_probs = [], [], [], []
    with torch.no_grad():
        for x_sig, x_feat, y in tqdm(loader, desc="Val", leave=False):
            x_sig  = x_sig.to(DEVICE)
            x_feat = x_feat.to(DEVICE)
            y      = y.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE == "cuda")):
                logits = model(x_sig, x_feat)
                loss   = criterion(logits, y)
            probs = F.softmax(logits, dim=1)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_trues.extend(y.cpu().numpy())
            all_losses.append(loss.item())
            all_probs.extend(probs.cpu().numpy())
    return compute_metrics(all_trues, all_preds, all_losses), np.array(all_probs), np.array(all_trues)


## 12 · Training loop

In [ ]:
start_epoch, best_val_f1, patience_ctr, history = load_checkpoint(
    model, optimizer, scheduler)

print(f"Starting from epoch {start_epoch + 1} / {MAX_EPOCHS}")
print(f"Early stopping patience: {PATIENCE} epochs")


/tmp/ipykernel_10261/2983903964.py:31: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Resumed from epoch 15  best val F1: 0.9102  scheduler step: 6240
Starting from epoch 16 / 50
Early stopping patience: 20 epochs


In [ ]:
for epoch in range(start_epoch, MAX_EPOCHS):

    tr_metrics = run_train_epoch(model, train_loader, optimizer, criterion,
                                 scheduler, amp_scaler)
    vl_metrics, vl_probs, vl_trues = run_val_epoch(model, val_loader, criterion)

    torch.cuda.empty_cache()

    lr      = optimizer.param_groups[0]["lr"]
    overfit = (tr_metrics["f1"] - vl_metrics["f1"]) > 0.10

    history["train"].append(tr_metrics)
    history["val"].append(vl_metrics)

    print_epoch(epoch + 1, MAX_EPOCHS, tr_metrics, vl_metrics, lr, overfit)

    if vl_metrics["f1"] > best_val_f1:
        best_val_f1  = vl_metrics["f1"]
        patience_ctr = 0
        save_checkpoint(model, optimizer, scheduler, amp_scaler,
                        epoch, best_val_f1, patience_ctr, history)
        print(f"  ✓ Saved checkpoint  best val F1: {best_val_f1:.4f}")
    else:
        patience_ctr += 1
        # Still save state so we can resume
        save_checkpoint(model, optimizer, scheduler, amp_scaler,
                        epoch, best_val_f1, patience_ctr, history)
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}  ({PATIENCE} epochs without improvement)")
            break

print(f"\nTraining complete.  Best val macro F1: {best_val_f1:.4f}")


Train:   0%|          | 0/416 [00:00<?, ?it/s]

Val:   0%|          | 0/74 [00:00<?, ?it/s]

Ep  16/50  lr=2.44e-04  | TR  loss=0.8186  F1=0.9380  F1_N=0.9373  F1_NN=0.9386  | VL  loss=1.2656  F1=0.9116  F1_N=0.9015  F1_NN=0.9218
  ✓ Saved checkpoint  best val F1: 0.9116


Train:   0%|          | 0/416 [00:00<?, ?it/s]

Val:   0%|          | 0/74 [00:00<?, ?it/s]

...

Ep  23/50  lr=1.82e-04  | TR  loss=0.7004  F1=0.9468  F1_N=0.9465  F1_NN=0.9472  | VL  loss=0.8785  F1=0.9381  F1_N=0.9298  F1_NN=0.9464
  ✓ Saved checkpoint  best val F1: 0.9381


Train:   0%|          | 0/416 [00:00<?, ?it/s]

Val:   0%|          | 0/74 [00:00<?, ?it/s]

Ep  24/50  lr=1.72e-04  | TR  loss=0.7177  F1=0.9463  F1_N=0.9466  F1_NN=0.9461  | VL  loss=0.8594  F1=0.9410  F1_N=0.9332  F1_NN=0.9488
  ✓ Saved checkpoint  best val F1: 0.9410


Train:   0%|          | 0/416 [00:00<?, ?it/s]

## 13 · Training curves

In [ ]:
with open(HISTORY_PATH) as f:
    history = json.load(f)

tr, vl = history["train"], history["val"]
epochs = range(1, len(tr) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metrics = [
    ("loss",  "Loss"),
    ("acc",   "Accuracy"),
    ("f1",    "Macro F1"),
    ("f1_N",  "Normal F1"),
    ("f1_NN", "NonNormal F1"),
    ("prec",  "Precision"),
]
for ax, (key, title) in zip(axes.flat, metrics):
    ax.plot(epochs, [m[key] for m in tr], label="train")
    ax.plot(epochs, [m[key] for m in vl], label="val")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "training_curves.png"), dpi=100)
plt.show()


## 14 · Overfitting gap

In [ ]:
gap = [t["f1"] - v["f1"] for t, v in zip(tr, vl)]
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(epochs, gap, color="tomato")
ax.axhline(0.10, color="gray", linestyle="--", label="overfit threshold (0.10)")
ax.set_title("Train - Val F1 gap"); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "overfit_gap.png"), dpi=100)
plt.show()


## 15 · Final evaluation on validation split

In [ ]:
model.load_state_dict(torch.load(CKPT_MODEL, map_location=DEVICE))
model.eval()

all_preds, all_trues, all_probs = [], [], []
with torch.no_grad():
    for x_sig, x_feat, y in tqdm(val_loader, desc="Final eval"):
        logits = model(x_sig.to(DEVICE), x_feat.to(DEVICE))
        probs  = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_trues.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)
all_probs = np.array(all_probs)

print("Classification report (best checkpoint, val split):")
print(classification_report(all_trues, all_preds,
      target_names=["Normal (0)", "NonNormal (1)"], digits=4))

macro_f1 = f1_score(all_trues, all_preds, average="macro")
print(f"Val macro F1 : {macro_f1:.4f}")
if macro_f1 >= 0.93:
    print("  ✓ Target met (≥ 0.93). Proceed to Stage 2.")
elif macro_f1 >= 0.90:
    print("  ✓ Good result. Fine-tune threshold before Stage 2.")
else:
    print("  ✗ Below target. Check class distribution and loss weights.")


## 16 · Confusion matrix

In [ ]:
cm = confusion_matrix(all_trues, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "NonNormal"])
ax.set_yticklabels(["Normal", "NonNormal"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Stage 1 confusion matrix (val, best checkpoint)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]}\n({cm[i,j]/cm[i].sum()*100:.1f}%)",
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=11)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

# Highlight specific failure modes
tn, fp, fn, tp = cm.ravel()
print(f"Normal correctly identified    : {tn}  ({tn/(tn+fp)*100:.1f}%)")
print(f"Normal misclassified as NonNorm: {fp}  ({fp/(tn+fp)*100:.1f}%)")
print(f"NonNormal missed (false Normal): {fn}  ({fn/(fn+tp)*100:.1f}%)")
print(f"NonNormal correctly identified : {tp}  ({tp/(fn+tp)*100:.1f}%)")


## 17 · Fine-grained threshold sweep

Sweeps the NonNormal probability threshold from 0.25 to 0.75 to find the decision boundary
that maximises macro F1 on the val split. The best threshold is saved for Stage 4 inference.


In [ ]:
def sweep_threshold(probs, trues, thresholds):
    results = []
    for thr in thresholds:
        preds = (probs[:, 1] >= thr).astype(int)
        f1    = f1_score(trues, preds, average="macro",   zero_division=0)
        f1_N  = f1_score(trues, preds, labels=[0], average="macro", zero_division=0)
        f1_NN = f1_score(trues, preds, labels=[1], average="macro", zero_division=0)
        prec  = precision_score(trues, preds, average="macro", zero_division=0)
        rec   = recall_score(trues,    preds, average="macro", zero_division=0)
        results.append({"threshold": thr, "f1": f1, "f1_N": f1_N,
                         "f1_NN": f1_NN, "prec": prec, "rec": rec})
    return results

thresholds  = np.arange(0.25, 0.76, 0.025).round(3)
thr_results = sweep_threshold(all_probs, all_trues, thresholds)

best = max(thr_results, key=lambda x: x["f1"])

print(f"{'Threshold':>10}  {'MacroF1':>9}  {'F1_N':>8}  {'F1_NN':>8}  {'Prec':>8}  {'Rec':>8}")
for r in thr_results:
    marker = "  ◄ best" if r["threshold"] == best["threshold"] else ""
    print(f"{r['threshold']:>10.3f}  {r['f1']:>9.4f}  {r['f1_N']:>8.4f}"
          f"  {r['f1_NN']:>8.4f}  {r['prec']:>8.4f}  {r['rec']:>8.4f}{marker}")

print(f"\nBest threshold : {best['threshold']}  macro F1: {best['f1']:.4f}"
      f"  F1_N: {best['f1_N']:.4f}  F1_NN: {best['f1_NN']:.4f}")

with open(os.path.join(CKPT_DIR, "best_threshold.json"), "w") as f:
    json.dump({"stage1_threshold": float(best["threshold"]),
               "macro_f1": float(best["f1"]),
               "f1_N": float(best["f1_N"]),
               "f1_NN": float(best["f1_NN"])}, f, indent=2)
print("Threshold saved to", os.path.join(CKPT_DIR, "best_threshold.json"))


### Threshold vs F1 plot

In [ ]:
thrs    = [r["threshold"] for r in thr_results]
f1s     = [r["f1"]        for r in thr_results]
f1_Ns   = [r["f1_N"]      for r in thr_results]
f1_NNs  = [r["f1_NN"]     for r in thr_results]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thrs, f1s,    label="Macro F1",    linewidth=2)
ax.plot(thrs, f1_Ns,  label="F1 Normal",   linestyle="--")
ax.plot(thrs, f1_NNs, label="F1 NonNormal",linestyle="--")
ax.axvline(best["threshold"], color="red", linestyle=":", label=f"Best thr={best['threshold']}")
ax.set_xlabel("Threshold (NonNormal probability)")
ax.set_ylabel("F1")
ax.set_title("Threshold sweep on validation split")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "threshold_sweep.png"), dpi=100)
plt.show()


## 18 · What is saved

| File | Contents |
|------|----------|
| `best_model.pt` | Best model weights (highest val macro F1) |
| `train_state.json` | Epoch, best F1, patience counter, scheduler state |
| `history.json` | Full per-epoch metrics for train and val |
| `best_threshold.json` | Tuned NonNormal threshold + F1 breakdown |
| `training_curves.png` | 6-panel metric curves |
| `overfit_gap.png` | Train−Val F1 gap |
| `confusion_matrix.png` | Best-checkpoint confusion matrix with percentages |
| `threshold_sweep.png` | Macro F1 / F1-N / F1-NN vs threshold |


In [ ]:
print("Stage 1 complete.")
print(f"  Best val macro F1 : {best_val_f1:.4f}")
print(f"  Best threshold    : {best['threshold']}")
print(f"  Normal F1         : {best['f1_N']:.4f}")
print(f"  NonNormal F1      : {best['f1_NN']:.4f}")
print("Proceed to Stage 2 (A vs O vs ~).")
